# Arricchimento Testi via EUR-Lex HTML

Questo notebook recupera il testo di ciascun atto normativo del grafo focale tramite scraping HTML diretto da EUR-Lex, usando **un solo metodo** per tutti gli atti (niente SPARQL).

## Cosa viene estratto

Per ogni atto vengono estratte **quattro sezioni** e concatenate nel campo `full_text_excerpt`:

| Sezione | Tag HTML | Perché è utile |
|---|---|---|
| **Titolo** | `p.oj-doc-ti` o `p.doc-ti` | Identifica chi ha adottato l'atto e con quale procedura |
| **Preambolo** | Prima `eli-subdivision`, fino al marker di fine | Base giuridica, obiettivi, procedura adottiva |
| **Primi 3 articoli** | Prime tre `eli-subdivision` con attributo article | Ambito applicativo, definizioni, obblighi principali |
| **Titoli degli allegati** | `eli-subdivision[type=annex]` — solo heading | Indica se l'atto è prevalentemente tecnico |

## Output

- `gephi_nodes_focal_texts.csv` — nodi con testo arricchito
- `texts_checkpoint.csv` — checkpoint per ripresa automatica

## Colonne aggiunte

| Colonna | Contenuto |
|---|---|
| `title` | Titolo completo dell'atto |
| `preamble` | Preambolo (considerando) |
| `articles_excerpt` | Testo dei primi 3 articoli |
| `annex_headings` | Titoli degli allegati, separati da ` | ` |
| `full_text_excerpt` | Concatenazione delle 4 sezioni, max 8000 caratteri |
| `sections_found` | Lista delle sezioni trovate (per diagnostica) |
| `text_status` | `ok` / `not_found` / `error` |
| `text_length` | Lunghezza di `full_text_excerpt` in caratteri |

## 0. Setup e Parametri

In [ ]:
import pandas as pd
import time
import os
import re
import sys
from bs4 import BeautifulSoup
import eurlex

sys.path.append('..')
from config_golden_power import MATERIA_NAME

# ── Percorsi ────────────────────────────────────────────────────────────────
output_path     = os.path.join('..', 'data', 'output', MATERIA_NAME)
input_file      = os.path.join(output_path, 'nodes_focal.csv')
output_file     = os.path.join(output_path, 'nodes_focal_texts.csv')
checkpoint_file = os.path.join(output_path, 'texts_checkpoint.csv')

# ── Parametri fetch ──────────────────────────────────────────────────────────
DELAY_SECONDS    = 0.7   # pausa tra richieste — rispetta il rate limit EUR-Lex
CHECKPOINT_EVERY = 50    # salva checkpoint ogni N nodi
TIMEOUT          = 20    # secondi per ciascuna richiesta HTTP

# ── Limiti di troncamento per sezione (caratteri) ───────────────────────────
# Nota: sentence-transformers tronca a ~2000 caratteri internamente.
# I limiti qui sotto servono per il CSV (visualizzazione, drill-down, LLM post-hoc).
MAX_TITLE_CHARS    = 400
MAX_PREAMBLE_CHARS = 8000    # preamboli lunghi hanno decine di considerando
MAX_ARTICLES_CHARS = 5000    # totale per i primi 3 articoli
MAX_ANNEX_CHARS    = 800     # solo intestazioni allegati
MAX_TOTAL_CHARS    = 15000   # limite finale full_text_excerpt
MAX_ARTICLES_N     = 3       # numero massimo di articoli da includere

# ── Marker fine preambolo ────────────────────────────────────────────────────
# Segnalano dove finisce il preambolo e inizia l'articolato
PREAMBLE_END_MARKERS = [
    'HAVE ADOPTED THIS REGULATION:',
    'HAS ADOPTED THIS REGULATION:',
    'HAVE ADOPTED THIS DIRECTIVE:',
    'HAS ADOPTED THIS DIRECTIVE:',
    'HAVE ADOPTED THIS DECISION:',
    'HAS ADOPTED THIS DECISION:',
    'HAVE ADOPTED THIS FRAMEWORK DECISION:',
    'HAVE ADOPTED THIS RECOMMENDATION:',
    'HEREBY DECIDES:',
    'HAS DECIDED AS FOLLOWS:',
    'HEREBY RECOMMENDS:',
    'IS OF THE OPINION THAT:',
    'HAVE AGREED AS FOLLOWS:',
    'HAVE DECIDED AS FOLLOWS:',
]

print(f"Input:      {input_file}")
print(f"Output:     {output_file}")
print(f"Checkpoint: {checkpoint_file}")

Input:      ..\data\output\golden_power\nodes_focal.csv
Output:     ..\data\output\golden_power\nodes_focal_texts.csv
Checkpoint: ..\data\output\golden_power\texts_checkpoint.csv


## 1. Caricamento Nodi e Gestione Checkpoint

In [ ]:
nodes = pd.read_csv(input_file)
print(f"Nodi totali nel grafo focale: {len(nodes)}")

CHECKPOINT_COLS = [
    'Id', 'Label',
    'title', 'preamble', 'articles_excerpt', 'annex_headings',
    'full_text_excerpt', 'sections_found',
    'text_status', 'text_length',
]

if os.path.exists(checkpoint_file):
    checkpoint   = pd.read_csv(checkpoint_file)
    already_done = set(checkpoint['Id'])
    print(f"Checkpoint trovato: {len(already_done)} nodi già processati")
    print(f"Nodi rimanenti:     {len(nodes) - len(already_done)}")
else:
    checkpoint   = pd.DataFrame(columns=CHECKPOINT_COLS)
    already_done = set()
    print("Nessun checkpoint trovato, si parte da zero")

nodes_todo = nodes[~nodes['Id'].isin(already_done)].copy()
print(f"Da processare ora: {len(nodes_todo)}")

Nodi totali nel grafo focale: 4904
Checkpoint trovato: 1800 nodi già processati
Nodi rimanenti:     3104
Da processare ora: 3104


## 2. Funzione di Fetch HTML

Un singolo metodo per tutti gli atti: richiesta HTTP diretta a EUR-Lex con il CELEX nell'URL.

Non viene usata la libreria `eurlex` perché nasconde il controllo sull'URL e sugli headers, rendendo il debug più difficile.

In [ ]:
# La libreria `eurlex` gestisce internamente la sessione e i cookie
# necessari per EUR-Lex, evitando il problema del 202 persistente.
# Usiamo SOLO il suo metodo di fetch — tutta la logica di estrazione
# del testo rimane la nostra.

import eurlex


def fetch_eurlex_html(celex, timeout=TIMEOUT):
    """
    Scarica la pagina HTML di un atto EUR-Lex dato il suo codice CELEX.

    Usa eurlex.get_html_by_celex_id() per il fetch (gestisce sessione
    e cookie EUR-Lex), poi valida che la struttura HTML sia riconoscibile.

    Restituisce (html_string, status) dove status è:
      'ok'           — HTML scaricato con struttura normativa riconoscibile
      'not_found'    — atto non trovato su EUR-Lex
      'no_structure' — HTML trovato ma senza tag normativi attesi
      'timeout'      — timeout della richiesta
      'error'        — errore generico
    """
    if pd.isna(celex) or str(celex).strip() == '':
        return None, 'not_found'

    celex = str(celex).strip()

    try:
        html = eurlex.get_html_by_celex_id(celex, language='en')

        if not html or len(html) < 500:
            return None, 'not_found'

        # Verifica che la pagina abbia struttura normativa riconoscibile
        STRUCTURAL_TAGS = ['eli-subdivision', 'oj-doc-ti', 'doc-ti']
        if not any(tag in html for tag in STRUCTURAL_TAGS):
            return None, 'no_structure'

        return html, 'ok'

    except Exception as e:
        err = str(e).lower()
        if 'timeout' in err:
            return None, 'timeout'
        if '404' in err or 'not found' in err:
            return None, 'not_found'
        return None, 'error'


# ── Test rapido ────────────────────────────────────────────────────────────
print("Test fetch su 3 atti con strutture diverse...")
print()
for celex_test in ['32019R0452', '32008L0114', '11957E']:
    html_test, status_test = fetch_eurlex_html(celex_test)
    size = f"{len(html_test):,} car." if html_test else '—'
    has_struct = ''
    if html_test:
        tags = [t for t in ['eli-subdivision', 'oj-doc-ti', 'doc-ti'] if t in html_test]
        has_struct = f"  tag trovati: {tags}"
    print(f"  {celex_test:<20} → {status_test:<20} {size}{has_struct}")
    time.sleep(1)

Test fetch su 3 atti con strutture diverse...

  32019R0452           → ok                   113,586 car.  tag trovati: ['eli-subdivision', 'oj-doc-ti', 'doc-ti']
  32008L0114           → ok                   71,175 car.  tag trovati: ['eli-subdivision', 'oj-doc-ti', 'doc-ti']
  11957E               → no_structure         —


## 3. Funzioni di Estrazione per Sezione

Ogni funzione accetta un oggetto `BeautifulSoup` già parsato e restituisce il testo estratto (o `None` se la sezione non è presente nella pagina).

In [ ]:
def extract_title(soup):
    """
    Estrae il titolo dell'atto dai tag p.oj-doc-ti o p.doc-ti.

    Combina più tag di titolo fino al primo che inizia con ANNEX/SCHEDULE/APPENDIX
    (che indica l'inizio degli allegati, non più il titolo dell'atto).
    """
    # Prova prima oj-doc-ti (formato moderno), poi doc-ti (formato precedente)
    for css_class in ['oj-doc-ti', 'doc-ti']:
        tags = soup.find_all('p', class_=css_class)
        if not tags:
            continue

        parti = []
        for tag in tags:
            testo = tag.get_text(separator=' ', strip=True)
            # Ferma prima di titoli di allegati
            if testo.startswith(('ANNEX', 'SCHEDULE', 'APPENDIX', 'ALLEGATO')):
                break
            parti.append(testo)

        if parti:
            return ' '.join(parti)[:MAX_TITLE_CHARS]

    # Fallback: tag <title> della pagina HTML
    title_tag = soup.find('title')
    if title_tag:
        text = title_tag.get_text(strip=True)
        # Rimuovi suffisso EUR-Lex
        text = re.sub(r'\s*[-–|]\s*EUR-Lex.*$', '', text, flags=re.IGNORECASE)
        if len(text) > 20:
            return text[:MAX_TITLE_CHARS]

    return None


def extract_preamble(soup):
    """
    Estrae il preambolo (considerando) dell'atto.

    Il preambolo si trova nella prima eli-subdivision della pagina.
    Viene troncato al primo marker che segnala l'inizio dell'articolato.
    """
    subdivisions = soup.find_all(class_='eli-subdivision')
    if not subdivisions:
        return None

    testo = subdivisions[0].get_text(separator=' ', strip=True)
    # Pulizia degli spazi multipli
    testo = re.sub(r'\s+', ' ', testo).strip()

    # Tronca al marker di fine preambolo
    for marker in PREAMBLE_END_MARKERS:
        idx = testo.find(marker)
        if idx != -1:
            testo = testo[:idx].strip()
            break

    if len(testo) < 50:  # troppo corto per essere un preambolo reale
        return None

    return testo[:MAX_PREAMBLE_CHARS]


def extract_articles(soup):
    """
    Estrae il testo dei primi MAX_ARTICLES_N articoli dell'atto.

    Strategia di ricerca in ordine decrescente di affidabilità:
    1. eli-subdivision con id che inizia con 'art_'
    2. eli-subdivision con attributo data-section='article'
    3. Sezioni con titolo che inizia con 'Article' (fallback)
    """
    article_divs = []

    # Strategia 1: id="art_1", "art_2" ecc.
    for div in soup.find_all(class_='eli-subdivision'):
        div_id = div.get('id', '')
        if re.match(r'^art_\d+', div_id):
            article_divs.append(div)

    # Strategia 2: data-section="article"
    if not article_divs:
        article_divs = soup.find_all(class_='eli-subdivision',
                                     attrs={'data-section': 'article'})

    # Strategia 3: fallback su titoli "Article X"
    if not article_divs:
        all_subdivs = soup.find_all(class_='eli-subdivision')
        for div in all_subdivs[1:]:  # salta il primo (preambolo)
            testo_raw = div.get_text(separator=' ', strip=True)[:100]
            if re.match(r'^Article\s+\d+', testo_raw, re.IGNORECASE):
                article_divs.append(div)

    if not article_divs:
        return None

    # Prendi i primi N articoli e concatena
    testi = []
    chars_used = 0
    per_article_limit = MAX_ARTICLES_CHARS // MAX_ARTICLES_N

    for div in article_divs[:MAX_ARTICLES_N]:
        testo = div.get_text(separator=' ', strip=True)
        testo = re.sub(r'\s+', ' ', testo).strip()
        testo = testo[:per_article_limit]
        testi.append(testo)
        chars_used += len(testo)

    if not testi:
        return None

    return ' \n\n '.join(testi)


def extract_annex_headings(soup):
    """
    Estrae solo i titoli (heading) degli allegati, non il corpo.

    Gli allegati tecnici sono spesso enormi — interessa solo sapere
    quanti ci sono e come si chiamano (indica il grado di tecnicità).
    """
    headings = []

    # Cerca tag con id che inizia con 'anx_' o 'ann_'
    annex_patterns = [r'^anx_', r'^ann_', r'^annex']
    for div in soup.find_all(class_='eli-subdivision'):
        div_id = div.get('id', '').lower()
        if any(re.match(p, div_id) for p in annex_patterns):
            # Prendi solo il primo tag di testo (il titolo)
            first_text = div.find(['p', 'div', 'span', 'h1', 'h2', 'h3'])
            if first_text:
                heading = first_text.get_text(strip=True)
                if heading and len(heading) > 2:
                    headings.append(heading[:100])

    # Fallback: cerca tag p con testo che inizia con ANNEX
    if not headings:
        for p in soup.find_all('p', class_=['oj-doc-ti', 'doc-ti', 'oj-ti-section']):
            text = p.get_text(strip=True)
            if text.startswith(('ANNEX', 'APPENDIX', 'SCHEDULE')):
                headings.append(text[:100])

    if not headings:
        return None

    result = ' | '.join(headings)
    return result[:MAX_ANNEX_CHARS]


print("Funzioni di estrazione definite.")

Funzioni di estrazione definite.


## 4. Funzione Principale: Estrazione Completa per Atto

In [ ]:
def extract_all_sections(celex):
    """
    Estrae tutte le sezioni di testo per un atto EUR-Lex dato il CELEX.

    Restituisce un dict con:
      title, preamble, articles_excerpt, annex_headings,
      full_text_excerpt, sections_found, text_status, text_length
    """
    empty = {
        'title':            None,
        'preamble':         None,
        'articles_excerpt': None,
        'annex_headings':   None,
        'full_text_excerpt': None,
        'sections_found':   '',
        'text_status':      None,
        'text_length':      0,
    }

    # Fetch HTML
    html, fetch_status = fetch_eurlex_html(celex)
    if fetch_status != 'ok':
        return {**empty, 'text_status': fetch_status}

    # Parse
    try:
        soup = BeautifulSoup(html, 'html.parser')
    except Exception:
        return {**empty, 'text_status': 'parse_error'}

    # Estrai sezioni
    title    = extract_title(soup)
    preamble = extract_preamble(soup)
    articles = extract_articles(soup)
    annexes  = extract_annex_headings(soup)

    # Registra quali sezioni sono state trovate
    sections_found = ','.join(filter(None, [
        'title'    if title    else None,
        'preamble' if preamble else None,
        'articles' if articles else None,
        'annexes'  if annexes  else None,
    ]))

    # Componi il testo completo con separatori espliciti
    parti = []
    if title:
        parti.append(f"[TITLE] {title}")
    if preamble:
        parti.append(f"[PREAMBLE] {preamble}")
    if articles:
        parti.append(f"[ARTICLES] {articles}")
    if annexes:
        parti.append(f"[ANNEXES] {annexes}")

    if not parti:
        return {**empty, 'text_status': 'no_content'}

    full_text = ' \n\n '.join(parti)[:MAX_TOTAL_CHARS]

    return {
        'title':             title,
        'preamble':          preamble,
        'articles_excerpt':  articles,
        'annex_headings':    annexes,
        'full_text_excerpt': full_text,
        'sections_found':    sections_found,
        'text_status':       'ok',
        'text_length':       len(full_text),
    }


print("Funzione principale definita.")

Funzione principale definita.


## 5. Test su Campione (eseguire prima del loop completo)

Verifica visivamente le sezioni estratte su 3 atti noti prima di lanciare il fetch completo.

In [ ]:
TEST_CELEX = [
    '32019R0452',   # Reg. FDI Screening — atto principale Golden Power
    '32008L0114',   # Dir. Infrastrutture critiche
    '62019CJ0079',  # Sentenza CGUE (diversa struttura HTML)
]

for celex in TEST_CELEX:
    print(f"{'='*60}")
    print(f"CELEX: {celex}")
    result = extract_all_sections(celex)
    print(f"  Status:         {result['text_status']}")
    print(f"  Sezioni trovate: {result['sections_found']}")
    print(f"  Lunghezza totale: {result['text_length']} caratteri")
    if result['title']:
        print(f"  Titolo:   {result['title'][:150]}")
    if result['preamble']:
        print(f"  Preambolo ({len(result['preamble'])} car., anteprima): {result['preamble'][:300]}")
    if result['articles_excerpt']:
        print(f"  Articoli  ({len(result['articles_excerpt'])} car., anteprima): {result['articles_excerpt'][:300]}")
    if result['annex_headings']:
        print(f"  Allegati: {result['annex_headings']}")
    print()
    time.sleep(1)

CELEX: 32019R0452
  Status:         ok
  Sezioni trovate: title,preamble,articles,annexes
  Lunghezza totale: 12440 caratteri
  Titolo:   REGULATION (EU) 2019/452 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 19 March 2019 establishing a framework for the screening of foreign direct i
  Preambolo (8000 car., anteprima): THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 207(2) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national p
  Articoli  (4208 car., anteprima): Article 1 Subject matter and scope 1. This Regulation establishes a framework for the screening by Member States of foreign direct investments into the Union on the grounds of security or public order and for a mechanism for cooperation between Member States, and between Member States and the Commis
  Allegati: ANNEX

CELEX: 32008L0114


## 6. Fetch Completo con Checkpoint

Con ~3000 nodi e 0.7s di delay il tempo stimato è circa **35 minuti**.

Se viene interrotto, riesegui questa cella: ripartirà automaticamente dal checkpoint.

In [ ]:
results  = []
total    = len(nodes_todo)
n_ok     = 0
n_err    = 0

print(f"Inizio fetch: {total} nodi")
print(f"Tempo stimato: ~{total * DELAY_SECONDS / 60:.0f} minuti\n")

for i, (_, row) in enumerate(nodes_todo.iterrows()):
    node_id = row['Id']
    celex   = row.get('Label', row.get('celex_id', ''))

    extracted = extract_all_sections(celex)

    if extracted['text_status'] == 'ok':
        n_ok += 1
    else:
        n_err += 1

    results.append({'Id': node_id, 'Label': celex, **extracted})

    # Progress ogni 10 nodi
    if (i + 1) % 10 == 0 or (i + 1) == total:
        pct = (i + 1) / total * 100
        print(f"  [{i+1:>4}/{total}] {pct:5.1f}%  ok: {n_ok}  errori: {n_err}")

    # Checkpoint periodico
    if (i + 1) % CHECKPOINT_EVERY == 0:
        batch              = pd.DataFrame(results)
        checkpoint_updated = pd.concat(
            [checkpoint, batch]
        ).drop_duplicates(subset=['Id'])
        checkpoint_updated.to_csv(checkpoint_file, index=False)
        print(f"  --> Checkpoint salvato ({len(checkpoint_updated)} nodi totali)")

    time.sleep(DELAY_SECONDS)

# Checkpoint finale
batch            = pd.DataFrame(results)
checkpoint_final = pd.concat(
    [checkpoint, batch]
).drop_duplicates(subset=['Id'])
checkpoint_final.to_csv(checkpoint_file, index=False)

print(f"\nFetch completato.")
print(f"  OK:        {(checkpoint_final['text_status'] == 'ok').sum()}")
print(f"  Not found: {(checkpoint_final['text_status'] == 'not_found').sum()}")
print(f"  Errori:    {checkpoint_final['text_status'].str.startswith('error', na=False).sum()}")

Inizio fetch: 3104 nodi
Tempo stimato: ~36 minuti

  [  10/3104]   0.3%  ok: 2  errori: 8
  [  20/3104]   0.6%  ok: 2  errori: 18
  [  30/3104]   1.0%  ok: 2  errori: 28
  [  40/3104]   1.3%  ok: 2  errori: 38
  [  50/3104]   1.6%  ok: 5  errori: 45
  --> Checkpoint salvato (1850 nodi totali)
  [  60/3104]   1.9%  ok: 15  errori: 45
  [  70/3104]   2.3%  ok: 25  errori: 45
  [  80/3104]   2.6%  ok: 35  errori: 45
  [  90/3104]   2.9%  ok: 45  errori: 45
  [ 100/3104]   3.2%  ok: 53  errori: 47
  --> Checkpoint salvato (1900 nodi totali)
  [ 110/3104]   3.5%  ok: 62  errori: 48
  [ 120/3104]   3.9%  ok: 71  errori: 49
  [ 130/3104]   4.2%  ok: 71  errori: 59
  [ 140/3104]   4.5%  ok: 71  errori: 69
  [ 150/3104]   4.8%  ok: 71  errori: 79
  --> Checkpoint salvato (1950 nodi totali)
  [ 160/3104]   5.2%  ok: 72  errori: 88
  [ 170/3104]   5.5%  ok: 72  errori: 98
  [ 180/3104]   5.8%  ok: 72  errori: 108
  [ 190/3104]   6.1%  ok: 72  errori: 118
  [ 200/3104]   6.4%  ok: 72  errori: 128


## CELLA 7: Definizione estrattori fallback

In [ ]:
import requests

SPARQL_ENDPOINT = 'https://publications.europa.eu/webapi/rdf/sparql'


# ── Strategia A: Template HTML legacy (atti ante-2000) ───────────────────────

def extract_legacy(celex):
    """
    Estrae testo da atti EUR-Lex con template HTML pre-ELI (ante ~2000).
    Questi atti non usano eli-subdivision ma div#TexteOnly / p.Normal.
    Fallback rispetto alla funzione extract_all_sections() già definita.
    """
    try:
        html = eurlex.get_html_by_celex_id(celex, language='en')
        if not html or len(html) < 300:
            return None
        soup = BeautifulSoup(html, 'html.parser')
    except Exception:
        return None

    # ── Titolo ────────────────────────────────────────────────────────────────
    title = None
    for css in ['Title', 'Titre', 'sti-tit', 'titredoc', 'doc-ti', 'oj-doc-ti']:
        tag = soup.find('p', class_=css)
        if tag:
            t = tag.get_text(separator=' ', strip=True)
            if len(t) > 20:
                title = t[:MAX_TITLE_CHARS]
                break
    if not title:
        h1 = soup.find('h1')
        if h1:
            t = h1.get_text(separator=' ', strip=True)
            title = t[:MAX_TITLE_CHARS] if len(t) > 20 else None
    if not title:
        tag = soup.find('title')
        if tag:
            t = re.sub(r'\s*[-–|]\s*EUR-Lex.*$', '', tag.get_text(strip=True), flags=re.IGNORECASE)
            title = t[:MAX_TITLE_CHARS] if len(t) > 20 else None

    # ── Corpo documento ───────────────────────────────────────────────────────
    container = None
    for sel in [{'id': 'TexteOnly'}, {'id': 'document1'}, {'id': 'docHtml'},
                {'class': 'texte'}, {'class': 'doc-content'}]:
        container = soup.find('div', sel)
        if container:
            break
    if not container:
        container = soup.find('body')
    if not container:
        return None

    for tag in container.find_all(['nav', 'header', 'footer', 'script', 'style', 'noscript']):
        tag.decompose()

    body = re.sub(r'\s+', ' ', container.get_text(separator=' ', strip=True)).strip()
    if len(body) < 100:
        return None

    # ── Split preambolo / articoli ────────────────────────────────────────────
    body_upper = body.upper()
    split_pos = None
    for marker in PREAMBLE_END_MARKERS:
        idx = body_upper.find(marker)
        if idx != -1:
            split_pos = idx
            break

    if split_pos:
        preamble     = body[:split_pos].strip()[:MAX_PREAMBLE_CHARS]
        articles_raw = body[split_pos:].strip()
    else:
        preamble     = body[:2000]
        articles_raw = body[2000:]

    # Estrai i primi 3 articoli
    art_matches = list(re.finditer(r'Article\s+\d+', articles_raw, re.IGNORECASE))
    articles = None
    if art_matches:
        start = art_matches[0].start()
        end   = art_matches[3].start() if len(art_matches) > 3 else len(articles_raw)
        articles = articles_raw[start:end].strip()[:MAX_ARTICLES_CHARS]

    preamble = preamble if preamble and len(preamble) > 50 else None
    articles = articles if articles and len(articles) > 20  else None

    sections = ','.join(filter(None, [
        'title'    if title    else None,
        'preamble' if preamble else None,
        'articles' if articles else None,
    ]))
    if not sections:
        return None

    parti = []
    if title:    parti.append(f"[TITLE] {title}")
    if preamble: parti.append(f"[PREAMBLE] {preamble}")
    if articles: parti.append(f"[ARTICLES] {articles}")
    full_text = ' \n\n '.join(parti)[:MAX_TOTAL_CHARS]

    return {
        'title':             title,
        'preamble':          preamble,
        'articles_excerpt':  articles,
        'annex_headings':    None,
        'full_text_excerpt': full_text,
        'sections_found':    sections,
        'text_status':       'ok',
        'text_length':       len(full_text),
    }


# ── Strategia B: Sentenze CGUE ────────────────────────────────────────────────

def extract_caselaw(celex):
    """
    Estrae testo da sentenze CGUE su EUR-Lex.
    Le sentenze usano p.C01Title, div#document1 e non hanno eli-subdivision.
    Il 'preambolo' corrisponde al contesto/legal framework,
    'articles_excerpt' corrisponde al dispositivo.
    """
    try:
        html = eurlex.get_html_by_celex_id(celex, language='en')
        if not html or len(html) < 300:
            return None
        soup = BeautifulSoup(html, 'html.parser')
    except Exception:
        return None

    # ── Titolo ────────────────────────────────────────────────────────────────
    title = None
    for css in ['C01Title', 'C01Titre', 'Title', 'oj-doc-ti', 'doc-ti', 'decision-title']:
        tags = soup.find_all('p', class_=css)
        if tags:
            t = ' '.join(tag.get_text(separator=' ', strip=True) for tag in tags[:3])
            if len(t) > 20:
                title = t[:MAX_TITLE_CHARS]
                break
    if not title:
        body_preview = soup.get_text(separator=' ', strip=True)[:500]
        m = re.search(
            r'(JUDGMENT OF THE COURT.*?\d{4}|ORDER OF THE COURT.*?\d{4}|'
            r'OPINION OF ADVOCATE GENERAL.*?\d{4})',
            body_preview, re.IGNORECASE
        )
        if m:
            title = m.group(0)[:MAX_TITLE_CHARS]
    if not title:
        tag = soup.find('title')
        if tag:
            t = re.sub(r'\s*[-–|]\s*EUR-Lex.*$', '', tag.get_text(strip=True), flags=re.IGNORECASE)
            title = t[:MAX_TITLE_CHARS] if len(t) > 15 else None

    # ── Corpo ─────────────────────────────────────────────────────────────────
    container = (
        soup.find('div', id='document1') or
        soup.find('div', id='TexteOnly') or
        soup.find('div', class_='decision') or
        soup.find('body')
    )
    if not container:
        return None

    for tag in container.find_all(['nav', 'header', 'footer', 'script', 'style']):
        tag.decompose()

    full = re.sub(r'\s+', ' ', container.get_text(separator=' ', strip=True)).strip()
    if len(full) < 100:
        return None

    # Split su marker del dispositivo
    DISPOSITIVO_MARKERS = [
        'ON THOSE GROUNDS, THE COURT',
        'HEREBY RULES:', 'GIVES THE FOLLOWING RULING:',
        'DECLARES AND RULES AS FOLLOWS:',
    ]
    full_upper = full.upper()
    split_pos = None
    for marker in DISPOSITIVO_MARKERS:
        idx = full_upper.find(marker)
        if idx != -1:
            split_pos = idx
            break

    if split_pos:
        preamble     = full[:split_pos].strip()[:MAX_PREAMBLE_CHARS]
        dispositivo  = full[split_pos:].strip()[:MAX_ARTICLES_CHARS]
    else:
        preamble     = full[:3000]
        dispositivo  = None

    preamble    = preamble    if preamble    and len(preamble)    > 50 else None
    dispositivo = dispositivo if dispositivo and len(dispositivo) > 50 else None

    sections = ','.join(filter(None, [
        'title'    if title       else None,
        'preamble' if preamble    else None,
        'articles' if dispositivo else None,
    ]))
    if not sections:
        return None

    parti = []
    if title:       parti.append(f"[TITLE] {title}")
    if preamble:    parti.append(f"[PREAMBLE] {preamble}")
    if dispositivo: parti.append(f"[DISPOSITIVO] {dispositivo}")
    full_text = ' \n\n '.join(parti)[:MAX_TOTAL_CHARS]

    return {
        'title':             title,
        'preamble':          preamble,
        'articles_excerpt':  dispositivo,
        'annex_headings':    None,
        'full_text_excerpt': full_text,
        'sections_found':    sections,
        'text_status':       'ok',
        'text_length':       len(full_text),
    }


# ── Strategia C: SPARQL Cellar per Treaty ─────────────────────────────────────

def extract_treaty_sparql(celex):
    """
    Recupera il testo di un articolo di trattato via SPARQL Cellar.
    L'endpoint è pubblico e gratuito (Publications Office EU).
    """
    query = f"""
    PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>
    SELECT DISTINCT ?title ?text
    WHERE {{
      ?work cdm:resource_legal_id_celex "{celex}"^^<http://www.w3.org/2001/XMLSchema#string> .
      OPTIONAL {{
        ?work cdm:work_title ?title .
        FILTER(lang(?title) = 'en' OR lang(?title) = '')
      }}
      OPTIONAL {{
        ?expr cdm:expression_belongs_to_work ?work .
        ?manif cdm:manifestation_manifests_expression ?expr .
        ?item cdm:item_belongs_to_manifestation ?manif .
        ?item <http://www.w3.org/1999/02/22-rdf-syntax-ns#value> ?text .
        FILTER(lang(?text) = 'en' OR lang(?text) = '')
      }}
    }}
    LIMIT 1
    """
    try:
        resp = requests.get(
            SPARQL_ENDPOINT,
            params={'query': query, 'format': 'application/sparql-results+json'},
            timeout=TIMEOUT,
            headers={'Accept': 'application/sparql-results+json'}
        )
        if resp.status_code != 200:
            return None

        bindings = resp.json().get('results', {}).get('bindings', [])
        if not bindings:
            return None

        title = bindings[0].get('title', {}).get('value')
        text  = bindings[0].get('text',  {}).get('value')

        if not title and not text:
            return None

        preamble = text[:MAX_PREAMBLE_CHARS] if text else None
        sections = ','.join(filter(None, [
            'title'    if title    else None,
            'preamble' if preamble else None,
        ]))
        if not sections:
            return None

        parti = []
        if title:    parti.append(f"[TITLE] {title}")
        if preamble: parti.append(f"[TEXT] {preamble}")
        full_text = ' \n\n '.join(parti)[:MAX_TOTAL_CHARS]

        return {
            'title':             title,
            'preamble':          preamble,
            'articles_excerpt':  None,
            'annex_headings':    None,
            'full_text_excerpt': full_text,
            'sections_found':    sections,
            'text_status':       'ok',
            'text_length':       len(full_text),
        }

    except Exception:
        return None


# ── Test rapido delle tre strategie ──────────────────────────────────────────
print("Test estrattori fallback...")
tests = [
    ('31999D0352', 'legacy',   extract_legacy),
    ('62008CJ0171', 'caselaw', extract_caselaw),
    ('12016E063',   'treaty',  extract_treaty_sparql),
]
for celex_t, label, fn in tests:
    r = fn(celex_t)
    status = r['text_status'] if r else 'None'
    sez    = r['sections_found'] if r else '—'
    lng    = r['text_length'] if r else 0
    print(f"  {label:<10} {celex_t:<15} → {status:<12} sezioni: {sez}  ({lng} car.)")
    time.sleep(1)


# Ricarica il checkpoint aggiornato dal fetch principale
checkpoint = pd.read_csv(checkpoint_file)

# Nodi da riprocessare: tutti quelli non 'ok' che hanno un CELEX valido
# Nota: i nodi 'not_found' genuini (CELEX inesistente su EUR-Lex) vengono
# comunque ritentati — alcune categorie (Treaty, Case_Law) erano not_found
# solo perché il metodo principale non gestiva il loro template.
failed_mask = checkpoint['text_status'] != 'ok'
failed_ids  = set(checkpoint[failed_mask]['Id'])

# Merge con i metadati per avere LegalType
if 'LegalType' not in checkpoint.columns:
    nodes_meta = pd.read_csv(input_file)    # nodes_focal.csv — ha LegalType
    checkpoint = checkpoint.merge(
        nodes_meta[['Id', 'LegalType']], on='Id', how='left'
    )

to_retry = checkpoint[checkpoint['Id'].isin(failed_ids)].copy()

print(f"Nodi da ritentare con fallback: {len(to_retry)}")
print(f"  Legacy  (altri tipi):  {len(to_retry[~to_retry['LegalType'].isin(['Treaty','Case_Law'])])}")
print(f"  CaseLaw:               {len(to_retry[to_retry['LegalType'] == 'Case_Law'])}")
print(f"  Treaty:                {len(to_retry[to_retry['LegalType'] == 'Treaty'])}")
print(f"\nTempo stimato: ~{len(to_retry) * DELAY_SECONDS / 60:.0f} minuti")

TEXT_COLS = [
    'title', 'preamble', 'articles_excerpt', 'annex_headings',
    'full_text_excerpt', 'sections_found', 'text_status', 'text_length'
]

n_recovered = n_still_failed = 0

for i, (idx, row) in enumerate(to_retry.iterrows()):
    celex      = str(row.get('Label', row['Id']))
    legal_type = row.get('LegalType', '')

    # Scegli estrattore
    if legal_type == 'Treaty':
        result = extract_treaty_sparql(celex)
    elif legal_type == 'Case_Law':
        result = extract_caselaw(celex)
    else:
        result = extract_legacy(celex)

    if result:
        # Aggiorna le colonne testuali nel checkpoint
        for col in TEXT_COLS:
            checkpoint.loc[checkpoint['Id'] == row['Id'], col] = result.get(col)
        n_recovered += 1
    else:
        n_still_failed += 1

    if (i + 1) % 10 == 0 or (i + 1) == len(to_retry):
        print(f"  [{i+1:>4}/{len(to_retry)}] {(i+1)/len(to_retry)*100:5.1f}%  "
              f"recuperati: {n_recovered}  ancora falliti: {n_still_failed}")

    # Salva checkpoint ogni N nodi
    if (i + 1) % CHECKPOINT_EVERY == 0:
        checkpoint.to_csv(checkpoint_file, index=False)
        print(f"  --> Checkpoint aggiornato")

    time.sleep(DELAY_SECONDS)

# Salvataggio finale del checkpoint aggiornato
checkpoint.to_csv(checkpoint_file, index=False)

print(f"\n{'='*50}")
print(f"FALLBACK COMPLETATO")
print(f"{'='*50}")
print(f"Recuperati con fallback: {n_recovered}")
print(f"Ancora senza testo:      {n_still_failed}")
print(f"\nNuova copertura nel checkpoint:")
print(checkpoint['text_status'].value_counts().to_string())

Test estrattori fallback...
  legacy     31999D0352      → ok           sezioni: title,preamble,articles  (6053 car.)
  caselaw    62008CJ0171     → ok           sezioni: preamble,articles  (8539 car.)
  treaty     12016E063       → None         sezioni: —  (0 car.)
Nodi da ritentare con fallback: 1667
  Legacy  (altri tipi):  1000
  CaseLaw:               220
  Treaty:                447

Tempo stimato: ~19 minuti
  [  10/1667]   0.6%  recuperati: 8  ancora falliti: 2
  [  20/1667]   1.2%  recuperati: 14  ancora falliti: 6
  [  30/1667]   1.8%  recuperati: 24  ancora falliti: 6
  [  40/1667]   2.4%  recuperati: 33  ancora falliti: 7
  [  50/1667]   3.0%  recuperati: 40  ancora falliti: 10
  --> Checkpoint aggiornato
  [  60/1667]   3.6%  recuperati: 49  ancora falliti: 11
  [  70/1667]   4.2%  recuperati: 54  ancora falliti: 16
  [  80/1667]   4.8%  recuperati: 60  ancora falliti: 20
  [  90/1667]   5.4%  recuperati: 69  ancora falliti: 21
  [ 100/1667]   6.0%  recuperati: 74  ancora 

## 8. Export File Finale

In [ ]:
texts_df = pd.read_csv(checkpoint_file)[[
    'Id', 'title', 'preamble', 'articles_excerpt',
    'annex_headings', 'full_text_excerpt',
    'sections_found', 'text_status', 'text_length',
]]

nodes_enriched = nodes.merge(texts_df, on='Id', how='left')
nodes_enriched.to_csv(output_file, index=False)

n_total = len(nodes_enriched)
n_ok    = (nodes_enriched['text_status'] == 'ok').sum()

print(f"File salvato: {output_file}")
print(f"  Nodi totali:  {n_total}")
print(f"  Con testo:    {n_ok} ({n_ok/n_total*100:.1f}%)")
print(f"  Senza testo:  {n_total - n_ok}")
print()
print("Lunghezza full_text_excerpt (caratteri):")
print(nodes_enriched['text_length'].describe().round(0).to_string())

File salvato: ..\data\output\golden_power\nodes_focal_texts.csv
  Nodi totali:  4904
  Con testo:    4225 (86.2%)
  Senza testo:  679

Lunghezza full_text_excerpt (caratteri):
count     4904.0
mean      6501.0
std       4335.0
min          0.0
25%       2595.0
50%       7038.0
75%      10342.0
max      13602.0


## 8. Diagnostica

Analisi dei nodi senza testo e distribuzione delle sezioni trovate.

In [ ]:
print("=" * 50)
print("DISTRIBUZIONE STATUS")
print("=" * 50)
print(nodes_enriched['text_status'].value_counts().to_string())
print()

print("=" * 50)
print("SEZIONI TROVATE (combinazioni più frequenti)")
print("=" * 50)
print(nodes_enriched['sections_found'].value_counts().head(10).to_string())
print()

# Copertura per tipo di atto
if 'LegalType' in nodes_enriched.columns:
    print("=" * 50)
    print("COPERTURA PER TIPO DI ATTO")
    print("=" * 50)
    cov = nodes_enriched.groupby('LegalType').agg(
        totale=('Id', 'count'),
        con_testo=('text_status', lambda x: (x == 'ok').sum())
    )
    cov['copertura_%'] = (cov['con_testo'] / cov['totale'] * 100).round(1)
    print(cov.sort_values('totale', ascending=False).to_string())
    print()

# Nodi senza testo — per capire cosa manca
no_text = nodes_enriched[nodes_enriched['text_status'] != 'ok']
if len(no_text) > 0:
    print("=" * 50)
    print(f"NODI SENZA TESTO: {len(no_text)}")
    print("=" * 50)
    if 'LegalType' in no_text.columns:
        print("Per tipo:")
        print(no_text['LegalType'].value_counts().to_string())
    if 'year' in no_text.columns:
        print("\nPer decade:")
        no_text_copy = no_text.copy()
        no_text_copy['decade'] = (no_text_copy['year'] // 10 * 10)
        print(no_text_copy['decade'].value_counts().sort_index().to_string())

DISTRIBUZIONE STATUS
text_status
ok              4225
no_structure     352
not_found        327

SEZIONI TROVATE (combinazioni più frequenti)
sections_found
title,preamble,articles            2237
title,preamble,articles,annexes    1760
title,preamble                       78
title                                46
preamble,articles                    32
title,preamble,annexes               31
preamble                             27
title,annexes                        14

COPERTURA PER TIPO DI ATTO
                   totale  con_testo  copertura_%
LegalType                                        
Decision             1685       1617         96.0
Regulation           1584       1576         99.5
Directive             611        608         99.5
Treaty                485         38          7.8
Case_Law              221        207         93.7
Legislative_Act       164         60         36.6
Recommendation        102         75         73.5
Guidelines             35         35        1

## 9. Verifica Qualità su Campione Random

Stampa 5 atti estratti con successo per verifica visiva della qualità del testo.

In [ ]:
sample = nodes_enriched[
    nodes_enriched['text_status'] == 'ok'
].sample(5, random_state=42)

for _, row in sample.iterrows():
    celex = row.get('Label', row['Id'])
    print(f"{'='*60}")
    print(f"CELEX: {celex}")
    print(f"Sezioni: {row['sections_found']} | Lunghezza: {row['text_length']} car.")
    print(f"Titolo: {str(row.get('title', ''))[:120]}")
    if pd.notna(row.get('preamble')):
        print(f"Preambolo (200c): {str(row['preamble'])[:200]}")
    if pd.notna(row.get('articles_excerpt')):
        print(f"Articoli (200c):  {str(row['articles_excerpt'])[:200]}")
    if pd.notna(row.get('annex_headings')):
        print(f"Allegati: {row['annex_headings']}")
    print()

CELEX: 32019R2175
Sezioni: title,preamble,articles | Lunghezza: 13444 car.
Titolo: REGULATION (EU) 2019/2175 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 18 December 2019 amending Regulation (EU) No 
Preambolo (200c): THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 114 thereof, Having regard to the proposa
Articoli (200c):  Article 1 Amendments to Regulation (EU) No 1093/2010 Regulation (EU) No 1093/2010 is amended as follows: (1) Article 1 is amended as follows: (a) paragraphs 2 and 3 are replaced by the following: ‘2. 

CELEX: 32018R1690
Sezioni: title,preamble,articles,annexes | Lunghezza: 11950 car.
Titolo: COMMISSION IMPLEMENTING REGULATION (EU) 2018/1690 of 9 November 2018 imposing definitive countervailing duties on import
Preambolo (200c): THE EUROPEAN COMMISSION, Having regard to the Treaty on the Functioning of the European Union, Having regard to Regulation (EU)